Correlation between dfs

In [ ]:
import pandas as pd

df1_path = r"C:\Users\papan\Desktop\Coffeomics\proteomics\data\prot_normalized_pydeseq2_manual_50_above_10.xlsx"
df2_path = r"C:\Users\papan\Desktop\Coffeomics\transcriptomics\trans_normalizations\trans_150bins_window7_normalized_counts.csv"
df3_path = r"C:\Users\papan\Desktop\Coffeomics\transcriptomics\trans_normalizations\trans_pydeseq2_normalized_counts.csv"
corr_path = r"C:\Users\papan\Desktop\Coffeomics\transcript_gene.csv"

prot = pd.read_excel(df1_path, index_col=0)
trans = pd.read_csv(df2_path, index_col=0)
trans_py = pd.read_csv(df3_path, index_col=0)
corr = pd.read_csv(corr_path, index_col=0)


In [ ]:
import json

with open(r"C:\Users\papan\Desktop\Coffeomics\proteomics\contrast_groups.json", "r") as f:
    contrast_groups = json.load(f)


In [ ]:
from scipy.stats import pearsonr

# 'transcript' is the index of corr, not a column — read it directly
transcript_to_gene = corr['gene']   # index is already the transcript ID

# Aggregate transcripts to gene level (mean across transcripts per gene)
trans_mapped = trans_py.copy()
trans_mapped.index = trans_mapped.index.map(transcript_to_gene)
trans_mapped = trans_mapped[trans_mapped.index.notna()]          # drop unmapped
trans_mapped = trans_mapped.groupby(trans_mapped.index).mean()   # collapse duplicates

# Keep only genes present in both omics layers
common_genes = prot.index.intersection(trans_mapped.index)
prot_aligned  = prot.loc[common_genes]
trans_aligned = trans_mapped.loc[common_genes]

# Common samples (columns) — should already match, but enforce it
common_samples = prot_aligned.columns.intersection(trans_aligned.columns)

# ── Diagnostic ────────────────────────────────────────────────────────────
only_in_prot  = set(prot_aligned.columns) - set(trans_aligned.columns)
only_in_trans = set(trans_aligned.columns) - set(prot_aligned.columns)

if only_in_prot:
    print(f"In proteomics only ({len(only_in_prot)}): {sorted(only_in_prot)}")
if only_in_trans:
    print(f"In transcriptomics only ({len(only_in_trans)}): {sorted(only_in_trans)}")
# ─────────────────────────────────────────────────────────────────────────

prot_aligned  = prot_aligned[common_samples]
trans_aligned = trans_aligned[common_samples]

# Per-sample Pearson correlation (columnwise), skipping NaNs pairwise
results = {}
for sample in common_samples:
    p = prot_aligned[sample]
    t = trans_aligned[sample]

    mask = p.notna() & t.notna()
    if mask.sum() >= 3:
        r, pval = pearsonr(p[mask], t[mask])
        results[sample] = {'pearson_r': r, 'pval': pval, 'n_genes': mask.sum()}
    else:
        results[sample] = {'pearson_r': float('nan'), 'pval': float('nan'), 'n_genes': mask.sum()}

results_df = pd.DataFrame(results).T
print(results_df)

In proteomics only (1): ['37D']
In transcriptomics only (1): ['37B']
     pearson_r          pval  n_genes
7A    0.393186  1.838680e-08    191.0
7B    0.039170  5.476156e-01    238.0
7C    0.464855  9.199785e-13    212.0
20B   0.436157  3.662851e-11    210.0
20C   0.510649  2.410651e-15    210.0
..         ...           ...      ...
52C   0.353949  7.837322e-08    218.0
52D   0.381327  1.833215e-08    204.0
54B   0.383985  7.425968e-09    212.0
54C   0.476373  1.696048e-12    196.0
54D   0.420590  1.537693e-10    213.0

[107 rows x 3 columns]


In [ ]:
import sys
from pathlib import Path

sys.path.append(
    str(Path(r"C:\Users\papan\Desktop\Coffeomics"))
)

from color_maps import COLOR_MAPS

In [ ]:
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
from scipy.stats import pearsonr
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT PATHS
# ─────────────────────────────────────────────────────────────────────────────

# Root folder where all outputs will be saved
OUTPUT_FOLDER = Path(r"C:\Users\papan\Desktop\Coffeomics\proteomics\correltion_matrix")

# Human-readable names for each data layer (used in plot titles and folder names)
FIRST_INPUT_DEF  = "Proteomics Normalized (manual, at least 50% and above 10)"
SECOND_INPUT_DEF = "Transcriptomics Normalized PyDESeq2"

# Create a subfolder named after the two inputs being compared
# safe_name() strips special characters so the path is valid on Windows

from make_correlations import safe_name
comparison_folder = OUTPUT_FOLDER / (
    f"{safe_name(FIRST_INPUT_DEF)}__vs__{safe_name(SECOND_INPUT_DEF)}"
)
# Create the folder (and any missing parents); do nothing if it already exists
comparison_folder.mkdir(parents=True, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# COLOR MAP — one color per contrast group
# ─────────────────────────────────────────────────────────────────────────────

# Ordered list of all contrast groups (keys of the contrast_groups dict)
unique_groups = list(contrast_groups.keys())

# Combine two 20-color matplotlib palettes → 40 distinct colors
# This handles datasets with many groups without color repetition
palette_colors = (
    list(plt.colormaps["tab20"].colors) +
    list(plt.colormaps["tab20b"].colors)
)

# Assign one color to each group, cycling through palette if > 40 groups
group_to_color = {
    g: palette_colors[i % len(palette_colors)]
    for i, g in enumerate(unique_groups)
}


# ─────────────────────────────────────────────────────────────────────────────
# SPECIES DETECTION
# ─────────────────────────────────────────────────────────────────────────────

# Extract unique species names from the group keys
# Group key format: "C. arabica | 25/20°C | 380ppm | without"
# → split on " | " and take the first element
all_species = sorted({g.split(" | ")[0] for g in unique_groups})
print(f"Detected species: {all_species}")

# Build a reverse lookup: short sample ID (e.g. "7A") → species name
# Needed later to assign each sample column to the correct species
short_id_to_species = {}
for group, short_ids in contrast_groups.items():
    species = group.split(" | ")[0]
    for sid in short_ids:
        short_id_to_species[sid] = species


# ─────────────────────────────────────────────────────────────────────────────
# SAMPLE → SPECIES ASSIGNMENT
# ─────────────────────────────────────────────────────────────────────────────

# common_samples: samples present in BOTH the proteomics and transcriptomics datasets
# Sort for reproducibility
sample_list = sorted(common_samples.tolist())

from make_correlations import col_to_species

# Map every sample to its species using col_to_species()
# col_to_species() tokenises the column name and matches against short_id_to_species
sample_to_species = {s: col_to_species(s, short_id_to_species) for s in sample_list}

# Warn if any samples could not be assigned to a species
# (likely means the short ID is missing from contrast_groups)
unmatched_samples = [s for s, sp in sample_to_species.items() if sp is None]
if unmatched_samples:
    print(f"Warning: {len(unmatched_samples)} sample(s) not assigned a species:")
    for u in unmatched_samples:
        print(f"  {u}")

# Group samples by species → {species_name: [sample_col, ...]}
species_samples = {
    sp: [s for s in sample_list if sample_to_species[s] == sp]
    for sp in all_species
}
# Report sample counts as a sanity check
for sp, slist in species_samples.items():
    print(f"  {sp}: {len(slist)} samples")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN BUILD LOOP — one clustermap set per species
# ─────────────────────────────────────────────────────────────────────────────
from make_correlations import build_and_save 

for sp, sp_samples in species_samples.items():
    if len(sp_samples) < 2:
        print(f"Skipping {sp}: fewer than 2 samples.")
        continue

    build_and_save(
        sp=sp,
        sp_samples=sp_samples,
        prot_aligned=prot_aligned,
        trans_aligned=trans_aligned,
        comparison_folder=comparison_folder,
        unique_groups=unique_groups,
        group_to_color=group_to_color,
        COLOR_MAPS=COLOR_MAPS,
        FIRST_INPUT_DEF=FIRST_INPUT_DEF,
        SECOND_INPUT_DEF=SECOND_INPUT_DEF,
        show_attribute_strips=True
    )

print("\nFinished.")

Detected species: ['C. arabica', 'C. canephora']
  C. arabica: 54 samples
  C. canephora: 53 samples

Building matrix for C. arabica (54 samples)...
  ✓  No comparisons excluded for C. arabica.


c:\Users\papan\Desktop\Coffeomics\layers_analysis\make_correlations.py:519: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  Saved: C:\Users\papan\Desktop\Coffeomics\proteomics\correltion_matrix\Proteomics_Normalized_manual_at_least_50_and_above_10__vs__Transcriptomics_Normalized_PyDESeq2\C_arabica_pearson_tree.png / .pdf
  Saved: C:\Users\papan\Desktop\Coffeomics\proteomics\correltion_matrix\Proteomics_Normalized_manual_at_least_50_and_above_10__vs__Transcriptomics_Normalized_PyDESeq2\C_arabica_pearson_clustermap.png / .pdf
  Saved: C:\Users\papan\Desktop\Coffeomics\proteomics\correltion_matrix\Proteomics_Normalized_manual_at_least_50_and_above_10__vs__Transcriptomics_Normalized_PyDESeq2\C_arabica_pearson_annotated_clustermap.png / .pdf
  Saved: C:\Users\papan\Desktop\Coffeomics\proteomics\correltion_matrix\Proteomics_Normalized_manual_at_least_50_and_above_10__vs__Transcriptomics_Normalized_PyDESeq2\C_arabica_pearson_ordered_heatmap.png / .pdf
  Saved: C:\Users\papan\Desktop\Coffeomics\proteomics\correltion_matrix\Proteomics_Normalized_manual_at_least_50_and_above_10__vs__Transcriptomics_Normalized_PyDES

c:\Users\papan\Desktop\Coffeomics\layers_analysis\make_correlations.py:519: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  Saved: C:\Users\papan\Desktop\Coffeomics\proteomics\correltion_matrix\Proteomics_Normalized_manual_at_least_50_and_above_10__vs__Transcriptomics_Normalized_PyDESeq2\C_canephora_pearson_tree.png / .pdf
  Saved: C:\Users\papan\Desktop\Coffeomics\proteomics\correltion_matrix\Proteomics_Normalized_manual_at_least_50_and_above_10__vs__Transcriptomics_Normalized_PyDESeq2\C_canephora_pearson_clustermap.png / .pdf
  Saved: C:\Users\papan\Desktop\Coffeomics\proteomics\correltion_matrix\Proteomics_Normalized_manual_at_least_50_and_above_10__vs__Transcriptomics_Normalized_PyDESeq2\C_canephora_pearson_annotated_clustermap.png / .pdf
  Saved: C:\Users\papan\Desktop\Coffeomics\proteomics\correltion_matrix\Proteomics_Normalized_manual_at_least_50_and_above_10__vs__Transcriptomics_Normalized_PyDESeq2\C_canephora_pearson_ordered_heatmap.png / .pdf
  Saved: C:\Users\papan\Desktop\Coffeomics\proteomics\correltion_matrix\Proteomics_Normalized_manual_at_least_50_and_above_10__vs__Transcriptomics_Normaliz

Sigle df correlation (againts itself, of it own features)

In [26]:
from make_correlations import build_and_save_single

'''
# ── Proteomics-only ───────────────────────────────────────────────────────────
PROT_DEF = "Proteomics Normalized (manual, at least 50% and above 10)"

prot_folder = OUTPUT_FOLDER / safe_name(PROT_DEF)
prot_folder.mkdir(parents=True, exist_ok=True)

for sp, sp_samples in species_samples.items():
    if len(sp_samples) < 2:
        continue
    build_and_save_single(
        sp=sp,
        sp_samples=sp_samples,
        data=prot_aligned,
        comparison_folder=prot_folder,
        unique_groups=unique_groups,
        group_to_color=group_to_color,
        COLOR_MAPS=COLOR_MAPS,
        INPUT_DEF=PROT_DEF,
    )

'''

# ── Transcriptomics-only ──────────────────────────────────────────────────────
TRANS_DEF = "Transcriptomics Normalized PyDESeq2"
OUTPUT_FOLDER = Path(r"C:\Users\papan\Desktop\Coffeomics\transcriptomics\correltion_matrix")

trans_folder = OUTPUT_FOLDER / safe_name(TRANS_DEF)
trans_folder.mkdir(parents=True, exist_ok=True)

for sp, sp_samples in species_samples.items():
    if len(sp_samples) < 2:
        continue
    build_and_save_single(
        sp=sp,
        sp_samples=sp_samples,
        data=trans_mapped,
        comparison_folder=trans_folder,
        unique_groups=unique_groups,
        group_to_color=group_to_color,
        COLOR_MAPS=COLOR_MAPS,
        INPUT_DEF=TRANS_DEF,
    )


Building matrix for C. arabica (54 samples)...
  ✓  No comparisons excluded for C. arabica.
  Saved: C:\Users\papan\Desktop\Coffeomics\transcriptomics\correltion_matrix\Transcriptomics_Normalized_PyDESeq2\C_arabica_pearson_tree.png / .pdf
  Saved: C:\Users\papan\Desktop\Coffeomics\transcriptomics\correltion_matrix\Transcriptomics_Normalized_PyDESeq2\C_arabica_pearson_clustermap.png / .pdf
  Saved: C:\Users\papan\Desktop\Coffeomics\transcriptomics\correltion_matrix\Transcriptomics_Normalized_PyDESeq2\C_arabica_pearson_annotated_clustermap.png / .pdf
  Saved: C:\Users\papan\Desktop\Coffeomics\transcriptomics\correltion_matrix\Transcriptomics_Normalized_PyDESeq2\C_arabica_pearson_ordered_heatmap.png / .pdf
  Saved: C:\Users\papan\Desktop\Coffeomics\transcriptomics\correltion_matrix\Transcriptomics_Normalized_PyDESeq2\C_arabica_gene_overlap_clustermap.png / .pdf
  Saved: C:\Users\papan\Desktop\Coffeomics\transcriptomics\correltion_matrix\Transcriptomics_Normalized_PyDESeq2\C_arabica_gene_